![Finance Toolkit](https://github.com/JerBouma/FinanceToolkit/assets/46355364/198d47bd-e1b3-492d-acc4-5d9f02d1d009)

**The FinanceToolkit** is an open-source toolkit in which all relevant financial ratios (100+), indicators and performance measurements are written down in the most simplistic way allowing for complete transparency of the calculation method. This allows you to not have to rely on metrics from other providers and, given a financial statement, allow for efficient manual calculations. This leads to one uniform method of calculation being applied that is available and understood by everyone.

# Installation
To install the FinanceToolkit it simply requires the following:

```
pip install financetoolkit -U
```

From within Python use:

```python
from financetoolkit import Toolkit
```
 
To be able to get started, you need to obtain an API Key from FinancialModelingPrep. This is used to gain access to 30+ years of financial statement both annually and quarterly. Note that the Free plan is limited to 250 requests each day, 5 years of data and only features companies listed on US exchanges.

___ 

<b><div align="center">Obtain an API Key from FinancialModelingPrep <a href="https://www.jeroenbouma.com/fmp" target="_blank">here</a>.</div></b>
___

Through the link you are able to subscribe for the free plan and also premium plans at a **15% discount**. This is an affiliate link and thus supports the project at the same time. I have chosen FinancialModelingPrep as a source as I find it to be the most transparent, reliable and at an affordable price. When you notice that data is inaccurate or have any other issue related to the data, note that I simply provide the means to access this data and I am not responsible for the accuracy of the data itself. For this, use <a href="https://site.financialmodelingprep.com/contact" target="_blank">their contact form</a> or provide the data yourself.

In [ ]:
import pandas as pd

from financetoolkit import Toolkit

API_KEY = "FINANCIAL_MODELING_PREP_API_KEY"

The Econometrics module provides a broad set of regression, hypothesis-testing, time-series and panel-data methods -- Ordinary Least Squares (OLS) and other regression estimators, unit root and cointegration tests, Granger causality, causal inference (IV/DiD/RDD/PSM/Synthetic Control), panel data (Fixed/Random Effects) and time-series forecasting (ARIMA, VAR, VECM). It is accessed through `toolkit.econometrics` and requires the optional `financetoolkit[econometrics]` extra (`pip install financetoolkit[econometrics]`), which pulls in `statsmodels` and `linearmodels`.

In [ ]:
# Initialize the Toolkit with company tickers
companies = Toolkit(
    ["MSFT", "AAPL", "AMZN", "META"], api_key=API_KEY, start_date="2013-01-01"
)

A natural starting point is an Ordinary Least Squares (OLS) regression, e.g. a CAPM-style regression of one asset's returns on a benchmark. Beyond the coefficient itself, OLS also reports the Standard Error, t-Statistic and P-Value for each regressor, letting you judge statistical significance rather than only the point estimate `performance.get_beta` gives you.

In [ ]:
companies.econometrics.get_ols(
    dependent_ticker="AAPL", independent_tickers=["Benchmark"], period="quarterly"
)

Before trusting a parametric Value at Risk model (e.g. the gaussian VaR in the `risk` module) it helps to test whether returns are actually normally distributed. The **Jarque-Bera test** combines sample skewness and excess kurtosis into a single statistic; a low p-value rejects normality.

In [ ]:
companies.econometrics.get_jarque_bera_test(period="quarterly", within_period=False)

Many time-series techniques (cointegration, several regressions) assume the underlying series is stationary. The **Augmented Dickey-Fuller (ADF) test** checks for a unit root -- its null hypothesis is that the series is *non-stationary*, so a low p-value (relative to the reported critical values) lets you reject that and treat the series as stationary.

In [ ]:
companies.econometrics.get_augmented_dickey_fuller(period="quarterly")

**Granger causality** tests whether one asset's lagged values help predict another's returns beyond that asset's own history -- evidence of a lead-lag relationship rather than true causation. **Engle-Granger cointegration** instead tests whether two price series share a common long-run equilibrium, the basis of many pairs-trading strategies.

In [ ]:
display(companies.econometrics.get_granger_causality(period="weekly"))

display(companies.econometrics.get_engle_granger_cointegration(period="weekly"))

Instead of comparing tickers pairwise, `get_fixed_effects` treats every ticker as an entity in a genuine panel and controls for time-invariant, entity-specific effects (e.g. a stock's typical risk premium) before estimating a shared regressor's coefficient -- here, sensitivity to the Benchmark.

In [ ]:
companies.econometrics.get_fixed_effects(
    independent_tickers="Benchmark", period="weekly"
)

Using the individual models with your own DataFrames is also a possibility thanks to the architecture of the Finance Toolkit.

In [ ]:
import numpy as np

from financetoolkit.econometrics import diagnostics_model, regression_model

np.random.seed(42)

factor = pd.Series(np.random.normal(0, 1, 100), name="Factor")
asset_return = 2.5 * factor + pd.Series(np.random.normal(0, 0.5, 100))
asset_return.name = "Asset Return"

result = regression_model.get_ols(asset_return, factor)

display(regression_model.regression_summary_table(result))

display(diagnostics_model.get_jarque_bera_test(asset_return))